# Phase 1 — Lesson 06: Probability and Distributions
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 06: Probability and Distributions
**Type:** Learn | **Languages:** Python | **Time:** ~75 minutes
**Prerequisites:** Phase 1, Lessons 01-04



# What You'll Learn

---

## Learning Objectives

- Implement PMFs and PDFs from scratch for Bernoulli, categorical, Poisson, uniform, and normal distributions
- Compute expected value, variance, and use the Central Limit Theorem to explain why Gaussians dominate
- Build softmax and log-softmax functions with the numerical stability trick (subtract max logit)
- Calculate cross-entropy loss from logits and connect it to negative log-likelihood


# The Problem

---

A classifier outputs `[0.03, 0.91, 0.06]`. A language model picks the next word from 50,000 candidates. A diffusion model generates images by sampling from learned distributions. All of these are probability in action.

Every prediction a model makes is a probability distribution. Every loss function measures how far the predicted distribution is from the true one.


# PMF vs PDF

---

| Type | Outcomes | Function | Examples |
|------|----------|----------|----------|
| Discrete | Countable | PMF | Bernoulli, categorical, Poisson |
| Continuous | Uncountable | PDF | Uniform, normal (Gaussian) |

Discrete: P(X = x) = pmf(x), sums to 1.  
Continuous: P(a ≤ X ≤ b) = ∫ₐᵇ pdf(x) dx; total area under curve = 1.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport mathdef bernoulli_pmf(k, p): return p if k == 1 else 1 - pdef categorical_pmf(k, probs): return probs[k]def poisson_pmf(k, lam): return (lam ** k) * math.exp(-lam) / math.factorial(k)def normal_pdf(x, mu, sigma):    coeff = 1 / (sigma * math.sqrt(2 * math.pi))    return coeff * math.exp(-0.5 * ((x - mu) / sigma) ** 2)b = bernoulli_pmf(1, 0.7)c = categorical_pmf(2, [0.1, 0.3, 0.6])p = poisson_pmf(2, 3.0)n = normal_pdf(0, 0, 1)print(f"Bernoulli P(1, p=0.7) = {b}")print(f"Categorical P(2) = {c}")print(f"Poisson P(2, λ=3) = {p:.4f}")print(f"Normal pdf(0, μ=0, σ=1) = {n:.4f}")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltx = np.linspace(-4, 4, 400)for mu, sig, c, lab in [(0,1,'#2c3e50','μ=0, σ=1'), (0,1.5,'#e74c3c','μ=0, σ=1.5'), (1,0.5,'#27ae60','μ=1, σ=0.5')]:    y = np.exp(-(x-mu)**2/(2*sig**2)) / (sig*np.sqrt(2*np.pi))    plt.plot(x, y, label=lab, color=c, lw=2)plt.axhline(0, color='gray', lw=0.8); plt.axvline(0, color='gray', lw=0.8)plt.title('Normal Distributions with Different μ and σ', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Expected Value and Variance

---

Expected value is the probability-weighted average outcome:
```
Discrete:   E[X] = sum of x_i * P(X = x_i)
Continuous: E[X] = integral of x * f(x) dx
```

Variance measures spread around the mean:
```
Var(X) = E[(X - E[X])²] = E[X²] - (E[X])²
Standard deviation = sqrt(Var(X))
```


In [ ]:
import numpy as npdie_values = np.array([1, 2, 3, 4, 5, 6])die_probs = np.array([1/6] * 6)mu = np.sum(die_values * die_probs)var = np.sum(die_probs * (die_values - mu) ** 2)print(f"Die: E[X] = {mu:.4f}, Var(X) = {var:.4f}, SD = {np.sqrt(var):.4f}")

# Softmax and Cross-Entropy

---

Neural networks output raw scores (logits). Softmax converts them into a valid probability distribution:

```
softmax(zᵢ) = exp(zᵢ) / sum(exp(zⱼ))
```

Cross-entropy loss for true class `t`:
```
L = -log(softmax(logits)[t])
```


In [ ]:
import mathdef softmax(logits):    max_logit = max(logits)    exps = [math.exp(z - max_logit) for z in logits]    total = sum(exps)    return [e / total for e in exps]def cross_entropy_loss(true_class, logits):    probs = softmax(logits)    return -math.log(probs[true_class])logits = [2.0, 1.0, 0.1]true_class = 0probs = softmax(logits)loss = cross_entropy_loss(true_class, logits)print(f"Logits: {logits}")print(f"Softmax: {[f'{p:.4f}' for p in probs]}")print(f"True class: {true_class}")print(f"Loss: {loss:.4f} nats")print(f"Perplexity: {math.exp(loss):.2f}")

# Central Limit Theorem

---

The Central Limit Theorem: the sum (or average) of many independent random variables converges to a normal distribution, regardless of the original distribution.

This works for ANY starting distribution. That is why:
- Measurement errors are approximately normal (many small independent sources)
- Weight initializations in neural networks use normal distributions
- Gradient noise in SGD is approximately normal


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport randomnp.random.seed(42)random.seed(42)# CLT: average of uniform samples becomes normaldef roll_die(): return random.randint(1, 6)averages_1 = [roll_die() for _ in range(10000)]averages_30 = [sum(random.randint(1, 6) for _ in range(30)) / 30 for _ in range(10000)]fig, axes = plt.subplots(1, 2, figsize=(10, 4))axes[0].hist(averages_1, bins=30, color='#95a5a6', alpha=0.7, edgecolor='white')axes[0].set_title('Single die roll (uniform)', fontweight='bold')axes[1].hist(averages_30, bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')axes[1].set_title('Average of 30 dice (≈ normal)', fontweight='bold')for ax in axes: ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print(f"Single die: mean={np.mean(averages_1):.3f}, std={np.std(averages_1):.3f}")print(f"Avg of 30:  mean={np.mean(averages_30):.3f}, std={np.std(averages_30):.3f}")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Difference between PMF and PDF?
> **Explanation:** PMF gives P(X=x) for discrete outcomes; PDF gives density for continuous outcomes (area = probability).

**Q2:** Why does the normal distribution show up everywhere in ML?
> **Explanation:** Central Limit Theorem — averages of many independent variables converge to normal.

**Q3:** What does cross-entropy loss measure?
> **Explanation:** Average surprise using model distribution Q to encode events from true distribution P.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Implement inverse transform sampling for the exponential distribution; verify histogram matches true PDF.

**Exercise 2:** Compute cross-entropy loss for logits [2.0, 0.5, -1.0, 3.0, 0.1] with true class 3.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| PMF | Probability mass function | P(X=x) for discrete variables |
| PDF | Probability density function | Density for continuous variables; area = probability |
| Bernoulli | Single binary trial | P(X=1) = p, P(X=0) = 1-p |
| Normal / Gaussian | Bell curve | Defined by mean μ and std σ |
| Cross-entropy | The loss function | -log(q(true_class)) for classification |


# Further Reading

---

- [Stanford CS229 Probability Review](https://cs229.stanford.edu/section/cs229-prob.pdf)
- [The Log-Sum-Exp Trick](https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/)


# Next Steps

---

When you're ready, continue to Lesson 07: Bayes' Theorem.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
